# TopoPerturb — Pilot Selection and PCA / Mapper (CausalBench)

**Purpose:** Follow the completed dataset audit by (1) selecting a transparent **eight-perturbation feasibility shortlist** from K562, (2) building a **shared PCA space** for controls and selected interventions, and (3) generating exploratory **Mapper graphs**. RPE1 labels are used to prioritize targets that can later be validated in the second cell line.

**Verified from your audit (not rechecked here):** K562 162,751 cells × 622 measured genes; RPE1 162,733 × 383 genes; last column `intervention`; controls `non-targeting`; `excluded` is **not** a control. Expression is already normalized and log-transformed. **Do not re-normalize.** There are no guide/batch covariates in these mirrors.

> This notebook **does not report scientific findings by itself**. It produces the shortlist, PCA visualizations, preliminary population statistics, and Mapper HTML files. The shortlist is based on cell-count coverage, not biological importance or final TDA significance. Formal PH/null tests are the next stage.

**Outputs:** `topoperturb_pilot_outputs/{k562,rpe1}_perturbation_counts.csv`, `shared_perturbations.csv`, `pilot_perturbations.csv`, `pilot_pca.png`, `pilot_pca_effect_summary.csv`, and `mapper_<gene_id>.html` for selected genes. All inputs are public Hugging Face mirrors.

## 0. Dependencies

In [ ]:
%pip install -q huggingface_hub pandas numpy scipy scikit-learn matplotlib kmapper

## 1. Configuration and downloads

For offline use, replace a download path in `LOCAL_DATA` with the exact location of your `.tsv.gz` file. Otherwise, the notebook downloads both public mirrors (roughly 168 MB compressed total).

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download
import gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
import kmapper as km

SEED = 42
OUT = Path("topoperturb_pilot_outputs")
OUT.mkdir(exist_ok=True)

REPOS = {
    "K562": ("huggingbahl21/causalbench-k562", "real/causalbench/data/causalbench_k562.tsv.gz"),
    "RPE1": ("huggingbahl21/causalbench-rpe1", "real/causalbench/data/causalbench_rpe1.tsv.gz"),
}
LOCAL_DATA = {"K562": None, "RPE1": None}  # Optional: local .tsv.gz paths

paths = {}
for cell_line, (repo, filename) in REPOS.items():
    if LOCAL_DATA[cell_line] is not None:
        paths[cell_line] = Path(LOCAL_DATA[cell_line])
    else:
        paths[cell_line] = Path(hf_hub_download(
            repo_id=repo, filename=filename, repo_type="dataset"
        ))
    if not paths[cell_line].exists():
        raise FileNotFoundError(paths[cell_line])
    print(f"{cell_line}: {paths[cell_line]}")

## 2. Verify header and count perturbation labels

This reads only the `intervention` column for counts; it does not load hundreds of expression columns into memory. We explicitly verify that `non-targeting` and `excluded` have the roles reported in the audit.

In [ ]:
gene_cols = {}
labels = {}
counts = {}

for line, path in paths.items():
    with gzip.open(path, "rt") as f:
        header = f.readline().rstrip("\n\r").split("\t")
    if header[-1] != "intervention":
        raise ValueError(f"{line}: final column is {header[-1]!r}, not 'intervention'")
    gene_cols[line] = header[:-1]
    if len(set(gene_cols[line])) != len(gene_cols[line]):
        raise ValueError(f"{line}: repeated gene-column names")

    labels[line] = pd.read_csv(path, sep="\t", compression="gzip",
                               usecols=["intervention"])["intervention"].astype("string")
    counts[line] = labels[line].value_counts(dropna=False)
    if labels[line].isna().any():
        raise ValueError(f"{line}: some cells have missing intervention labels")

    measured = set(gene_cols[line])
    target_counts = counts[line][counts[line].index.isin(measured)]
    out = pd.DataFrame({"intervention": counts[line].index.astype(str),
                        "n_cells": counts[line].values})
    out["is_control"] = out.intervention.eq("non-targeting")
    out["is_excluded"] = out.intervention.eq("excluded")
    out.to_csv(OUT / f"{line.lower()}_perturbation_counts.csv", index=False)

    print(f"\n{line}: {len(labels[line]):,} cells; {len(gene_cols[line])} measured genes")
    print("Controls:", int(counts[line].get("non-targeting", 0)),
          " Excluded:", int(counts[line].get("excluded", 0)))
    print("Targeted genes measured:", len(target_counts))
    print("Targets >=200 cells:", int((target_counts >= 200).sum()))
    extra = set(counts[line].index.astype(str)) - measured - {"non-targeting", "excluded"}
    if extra:
        print("WARNING: unexpected intervention labels:", sorted(extra)[:20])

## 3. Find shared targets and select a balanced-feasibility K562 pilot

**Selection is predetermined by metadata, before inspecting PCA/TDA results:** select up to four targets shared with RPE1 and up to four K562-only targets, spanning the lower/middle/upper parts of the eligible cell-count range. Prefer K562 ≥200 cells and shared RPE1 ≥150; relax automatically if there are too few eligible genes. This avoids selecting targets after seeing the desired topological result. It is **not** a biological-target prioritization method.

In [ ]:
def target_table(line):
    genes = set(gene_cols[line])
    s = counts[line][counts[line].index.isin(genes)]
    return pd.DataFrame({"gene_id": s.index.astype(str),
                         f"n_{line.lower()}": s.to_numpy(dtype=int)})

k = target_table("K562")
r = target_table("RPE1")
shared = k.merge(r, on="gene_id", how="inner").sort_values("n_k562", ascending=False)
shared.to_csv(OUT / "shared_perturbations.csv", index=False)
print(f"Shared measured-and-targeted genes: {len(shared)}")

def spread_pick(df, n, count_col="n_k562"):
    """Deterministic count-stratified pick with no repeated IDs."""
    df = df.sort_values([count_col, "gene_id"]).reset_index(drop=True)
    if df.empty or n <= 0:
        return df.iloc[:0].copy()
    n = min(n, len(df))
    # Select equally spaced count ranks, avoiding duplicates in small tables.
    ranks = np.linspace(0, len(df)-1, n).round().astype(int)
    return df.iloc[np.unique(ranks)].copy()

shared_candidates = shared[(shared.n_k562 >= 200) & (shared.n_rpe1 >= 150)]
if len(shared_candidates) < 4:
    shared_candidates = shared[(shared.n_k562 >= 150) & (shared.n_rpe1 >= 100)]

k_only = k[~k.gene_id.isin(set(shared.gene_id))]
k_only_candidates = k_only[k_only.n_k562 >= 200]
if len(k_only_candidates) < 4:
    k_only_candidates = k_only[k_only.n_k562 >= 150]

pilot_shared = spread_pick(shared_candidates, min(4,len(shared_candidates)))
pilot_only = spread_pick(k_only_candidates, min(4,len(k_only_candidates)))
pilot_shared["group"] = "shared_RPE1"
pilot_only["group"] = "K562_only"

pilot = pd.concat([pilot_shared, pilot_only], ignore_index=True)
# If fewer than 8 eligible, fill from other K562 targets (>=100 by audit).
if len(pilot) < 8:
    remaining = k[~k.gene_id.isin(set(pilot.gene_id)) & (k.n_k562 >= 100)]
    filler = spread_pick(remaining, 8-len(pilot))
    filler["group"] = "additional_K562"
    pilot = pd.concat([pilot, filler], ignore_index=True)
pilot = pilot.merge(r, on="gene_id", how="left", suffixes=("", "_lookup"))
if "n_rpe1_lookup" in pilot.columns:
    pilot["n_rpe1"] = pilot.get("n_rpe1", pilot["n_rpe1_lookup"]).fillna(pilot["n_rpe1_lookup"])
    pilot = pilot.drop(columns=["n_rpe1_lookup"])
pilot["n_rpe1"] = pilot["n_rpe1"].fillna(0).astype(int)
pilot = pilot[["gene_id", "n_k562", "n_rpe1", "group"]]
assert pilot.gene_id.is_unique
pilot.to_csv(OUT / "pilot_perturbations.csv", index=False)
print("\nPilot shortlist (gene IDs are Ensembl identifiers):")
print(pilot.to_string(index=False))
print("\nSaved:", OUT / "pilot_perturbations.csv")

## 4. Load only control + pilot cells from K562

The TSV is still parsed once in chunks, but only the controls and chosen perturbations are retained. `excluded` cells are *never* included. This is less memory-intensive than keeping the whole dense dataset. Do not include an independent normalization step; these values are already preprocessed.

In [ ]:
pilot_ids = pilot.gene_id.tolist()
keep = set(pilot_ids) | {"non-targeting"}
dtypes = {g: np.float32 for g in gene_cols["K562"]}
dtypes["intervention"] = "string"

parts = []
for chunk in pd.read_csv(paths["K562"], sep="\t", compression="gzip",
                         dtype=dtypes, chunksize=2500):
    take = chunk["intervention"].isin(keep)
    if take.any():
        parts.append(chunk.loc[take].copy())

pilot_data = pd.concat(parts, ignore_index=True)
assert not pilot_data.intervention.isin(["excluded"]).any()
print("Retained rows:", len(pilot_data))
print(pilot_data.intervention.value_counts().to_string())
assert set(pilot_ids).issubset(set(pilot_data.intervention))

## 5. Common PCA space and exploratory visualization

Sample equal numbers of cells per group for the visualization. Fit **one shared PCA** to the selected K562 groups. We use mean centering (part of PCA), **not** a second library normalization or log transform. The seed and gene list are recorded to make the pilot repeatable.

In [ ]:
N_PER_GROUP = min(150, int(pilot_data.intervention.value_counts().loc[pilot_ids].min()))
if N_PER_GROUP < 30:
    raise ValueError("Too few cells per group for a useful pilot")

sampled = []
for j, label in enumerate(["non-targeting", *pilot_ids]):
    sub = pilot_data.loc[pilot_data.intervention == label]
    sampled.append(sub.sample(n=N_PER_GROUP, random_state=SEED+j, replace=False))
balanced = pd.concat(sampled, ignore_index=True)

X = balanced.loc[:, gene_cols["K562"]].to_numpy(dtype=np.float32, copy=True)
if not np.isfinite(X).all():
    raise ValueError("Nonfinite values found in expression matrix")
Y = balanced.intervention.astype(str).to_numpy()
n_components = min(30, X.shape[1], X.shape[0]-1)
pca = PCA(n_components=n_components, svd_solver="randomized", random_state=SEED)
Z = pca.fit_transform(X)

plt.figure(figsize=(8.5, 6))
focus = ["non-targeting", *pilot_ids[:3]]
for label in focus:
    loc = Y == label
    plt.scatter(Z[loc,0], Z[loc,1], s=12, alpha=0.65, label=label)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title(f"CausalBench K562 pilot: balanced PCA, n={N_PER_GROUP}/group")
plt.legend(title="Intervention", fontsize=8)
plt.tight_layout()
plt.savefig(OUT / "pilot_pca.png", dpi=220)
plt.show()
print("PCA shape:", Z.shape, "Explained variance (first 10):",
      round(float(pca.explained_variance_ratio_[:10].sum()), 3))

## 6. Preliminary conventional effect-size summary

This table is **descriptive only**, not a differential-expression result or statistical test. It can help identify perturbations with similar centroid shift but different dispersion for a later matched comparison. Both measures are computed in the **same PCA space**. No topological novelty is claimed at this stage.

In [ ]:
z_ctrl = Z[Y == "non-targeting"]
c_ctrl = z_ctrl.mean(axis=0)
s_ctrl = np.mean(np.linalg.norm(z_ctrl - c_ctrl, axis=1))
rows=[]
for gene in pilot_ids:
    z = Z[Y == gene]
    c = z.mean(axis=0)
    rows.append({"gene_id":gene,
                 "n_pilot_cells":len(z),
                 "centroid_shift_l2":float(np.linalg.norm(c-c_ctrl)),
                 "mean_radial_dispersion":float(np.mean(np.linalg.norm(z-c,axis=1))),
                 "dispersion_ratio_to_control":float(np.mean(np.linalg.norm(z-c,axis=1))/(s_ctrl+1e-12)),
                 "n_rpe1_available":int(pilot.set_index("gene_id").loc[gene,"n_rpe1"])})
effects=pd.DataFrame(rows).sort_values("centroid_shift_l2", ascending=False)
effects.to_csv(OUT / "pilot_pca_effect_summary.csv", index=False)
print(effects.round(3).to_string(index=False))

## 7. Exploratory Mapper networks

Generate **two-group combined Mapper graphs** (control + one perturbation at a time) for the first three selected genes. The lens is the first two PCs of the *shared* PCA; clustering uses ten PCs. The algorithm chooses a rough DBSCAN radius using local-neighbor distances. **Mapper graphs are exploratory and depend on hyperparameters.** Nodes are not automatically biological cell states, and branch-like structures do not imply trajectories.

In [ ]:
mapper = km.KeplerMapper(verbose=0)
HTML_GENES = pilot_ids[:3]  # change after inspecting pilot results

for gene in HTML_GENES:
    mask = np.isin(Y, ["non-targeting", gene])
    z = Z[mask]
    y = Y[mask]
    # kNN-derived heuristic radius; inspect sensitivity before any scientific claim.
    embed = z[:, :min(10, z.shape[1])]
    nn = NearestNeighbors(n_neighbors=min(8, len(embed)-1)).fit(embed)
    dists, _ = nn.kneighbors(embed)
    eps = max(float(np.quantile(dists[:,-1], 0.80)) * 1.25, 1e-6)
    graph = mapper.map(
        lens=z[:, :2],
        X=embed,
        cover=km.Cover(n_cubes=8, perc_overlap=0.4),
        clusterer=DBSCAN(eps=eps, min_samples=4),
    )
    output_path = OUT / f"mapper_{gene}.html"
    mapper.visualize(graph, path_html=str(output_path),
                     title=f"K562 control versus {gene}",
                     color_values=(y == gene).astype(float),
                     color_function_name=f"Perturbed ({gene}) fraction")
    num_nodes = len(graph["nodes"])
    print(f"{gene}: {num_nodes} Mapper nodes; eps={eps:.3f}; saved {output_path}")
    if num_nodes < 3:
        print("  CAUTION: sparse graph; modify cover / DBSCAN and assess stability.")

## 8. How to interpret and what to do next

**Report these outputs first:** (a) `pilot_perturbations.csv`; (b) `pilot_pca_effect_summary.csv`; (c) `pilot_pca.png`; and (d) the HTML Mapper graphs, or screenshots of them.

**Next experiments, not performed in this notebook:**

1. Pre-register perturbations and comparison statistics before checking persistent homology results.
2. Compute H0/H1 diagrams using `ripser` on equal-size independently sampled groups. For 100–150 cells/group, start with 10–30 PCs and test sensitivity to dimensions.
3. Build a control-versus-control reference distribution with disjoint control subsamples and repeated draws; run label-permutation tests and correct for multiple perturbations.
4. Compare Wasserstein/bottleneck persistence distances with conventional mean shifts, dispersion, and full-distribution distances such as MMD or energy distance; test reproducibility across sampling.
5. If testing effective resistance/diffusion distances, use a clearly defined comparable construction; these metrics can be sensitive to graph choices. Check that scales are consistent between groups.
6. Validate shared targets in RPE1; fit each cell line independently, never directly compare raw PCA coordinates or persistence scales across lines.

**Known limitation:** These public mirrors include only `intervention` labels, not per-guide or batch information. Equal cell counts alone do *not* control batch or guide confounding, and causal claims about topology must remain limited accordingly.

**Useful reminder:** Gene labels are Ensembl IDs. Map to symbols with a versioned annotation source before interpreting biological mechanisms; do not infer a gene symbol from an identifier.
